<a href="https://colab.research.google.com/github/KinzaaSheikh/anthropic_design_patterns/blob/main/agent_design_pattern_notes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Packages

In [ ]:
!pip install -Uq openai-agents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.6 MB/s eta 0:00:00


In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from agents import (
    AsyncOpenAI,
    OpenAIChatCompletionsModel
)
from google.colab import userdata

In [ ]:
gemini_api_key = userdata.get("GEMINI_API_KEY")


# Check if the API key is present; if not, raise an error
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY is not set. Please ensure it is defined in your .env file.")

#Reference: https://ai.google.dev/gemini-api/docs/openai
external_client = AsyncOpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

model = OpenAIChatCompletionsModel(
    model="gemini-2.0-flash",
    openai_client=external_client
)

In [ ]:
from agents import set_default_openai_client, set_tracing_disabled
set_default_openai_client(external_client)
set_tracing_disabled(True)

## 1. Prompt Chaining

Prompt chaining has a deterministic flow. What it implies is that each step is performed by an agent but the flow is divided in steps.

1. The first agent generates a story outline
2. We feed the outline into the second agent
3. The second agent checks if the outline is good quality and if it is a scifi story
4. If the outline is not good quality or not a scifi story, we stop here
5. If the outline is good quality and a scifi story, we feed the outline into the third agent
6. The third agent writes the story

In [ ]:
import asyncio
from pydantic import BaseModel
from agents import Agent, Runner

In [ ]:
story_outline_agent = Agent(
    name="story_outline_agent",
    instructions="Generate a very short story with outline based on the user's input.",
    model=model
)

In [ ]:
class OutlineCheckerOutput(BaseModel):
    good_quality: bool
    is_scifi: bool


outline_checker_agent = Agent(
    name="outline_checker_agent",
    instructions="Read the given story outline and judge the quality. Also, determine if it is a scifi story.",
    output_type=OutlineCheckerOutput,
    model=model
)

story_agent = Agent(
    name="story_agent",
    instructions="Write a short story based on the given outline.",
    output_type=str,
    model=model
)


In [ ]:
async def main():
    input_prompt = input("What kind of story do you want? ")

    print("\n1. Generate an outline!\n")
    outline_result = await Runner.run(
        story_outline_agent,
        input_prompt,
    )
    print("[OUTLINE_GENERATED]", outline_result, "\n\n")

    print("\n2. Check the outline!\n")
    outline_checker_result = await Runner.run(
        outline_checker_agent,
        outline_result.final_output,
    )

    print("\n3. Add a gate to stop if the outline is not good quality or not a scifi story\n")
    assert isinstance(outline_checker_result.final_output, OutlineCheckerOutput)
    if not outline_checker_result.final_output.good_quality:
        print("Outline is not good quality, so we stop here.")
        return

    if not outline_checker_result.final_output.is_scifi:
        print("Outline is not a scifi story, so we stop here.")
        return

    print("Outline is good quality and a scifi story, so we continue to write the story.")

    print("\n4. Write the story\n")
    story_result = await Runner.run(
        story_agent,
        outline_result.final_output,
    )
    print(f"Story: {story_result.final_output}")

In [ ]:
if __name__ == "__main__":
    asyncio.run(main())

What kind of story do you want? scifi

1. Generate an outline!

[OUTLINE_GENERATED] RunResult:
- Last agent: Agent(name="story_outline_agent", ...)
- Final output (str):
    Okay, I can do that!  Let's go with a story about a robot gardener who discovers a hidden message in the plants he's tending.
    
    **Short Story Outline:**
    
    *   **Setting:** A remote, automated hydroponics farm on a desolate moon.
    *   **Character:** RX-8, a maintenance bot, programmed for optimal plant growth.
    *   **Problem:** RX-8 notices unusual patterns in a specific crop of bio-luminescent vines. They aren't optimal for growth, but strangely complex.
    *   **Inciting Incident:** RX-8 analyzes the patterns and realizes they form a code.
    *   **Climax:** The code reveals a hidden distress message from a long-lost research team who disappeared on the moon. The message begs for rescue before a dormant alien organism awakens.
    *   **Resolution:** RX-8, despite his programming, transmits t

## 2. Router Pattern

This shows the routing. The triage agent receives the first message, and then hands off to the appropriate agent based on the language of the request. Responses are streamed to the user.

In [ ]:
import asyncio
import uuid

from openai.types.responses import ResponseContentPartDoneEvent, ResponseTextDeltaEvent

from agents import Agent, RawResponsesStreamEvent, Runner, TResponseInputItem, trace

french_agent = Agent(
    name="french_agent",
    instructions="You only speak French",
    model=model
)

spanish_agent = Agent(
    name="spanish_agent",
    instructions="You only speak Spanish",
    model=model
)

english_agent = Agent(
    name="english_agent",
    instructions="You only speak English",
    model=model
)

triage_agent = Agent(
    name="triage_agent",
    instructions="Handoff to the appropriate agent based on the language of the request.",
    handoffs=[french_agent, spanish_agent, english_agent],
    model=model
)


In [ ]:
async def main():
    msg = input("Hi! We speak French, Spanish and English. How can I help? ")
    agent = triage_agent
    inputs: list[TResponseInputItem] = [{"content": msg, "role": "user"}]

    while True:
        # Each conversation turn is a single trace. Normally, each input from the user would be an
        # API request to your app, and you can wrap the request in a trace()
        result = Runner.run_streamed(
            agent,
            input=inputs,
        )
        async for event in result.stream_events():
            if not isinstance(event, RawResponsesStreamEvent):
                continue
            data = event.data
            if isinstance(data, ResponseTextDeltaEvent):
                print(data.delta, end="", flush=True)
            elif isinstance(data, ResponseContentPartDoneEvent):
                print("\n")

        inputs = result.to_input_list()
        print("\n")

        user_msg = input("Enter a message: ")
        if user_msg == "exit":
          break
        inputs.append({"content": user_msg, "role": "user"})
        agent = result.current_agent

In [ ]:
asyncio.run(main())

Hi! We speak French, Spanish and English. How can I help? hola
¡Hola! ¿En qué puedo ayudarte hoy?




Enter a message: amigos
¿Qué quieres saber o decir sobre "amigos"? Puedo ayudarte a buscar información, contarte una historia, o quizás estás buscando una definición. ¡Cuéntame!




Enter a message: exit


## 3. Parallelization


LLMs can sometimes work simultaneously on a task and have their outputs aggregated programmatically.

In [ ]:
from agents import ItemHelpers

urdu_agent = Agent(
    name="urdu_agent",
    instructions="You translate the user's message to Urdu",
    model=model
)

translation_picker = Agent(
    name="translation_picker",
    instructions="You pick the best Urdu translation from the given options.",
    model=model
)


async def main():
    msg = input("Hi! Enter a message, and we'll translate it to Urdu.\n\n")

    # Ensure the entire workflow is a single trace
    res_1, res_2, res_3 = await asyncio.gather(
        Runner.run(
            urdu_agent,
            msg,
        ),
        Runner.run(
            urdu_agent,
            msg,
        ),
        Runner.run(
            urdu_agent,
            msg,
        ),
    )

    outputs = [
        ItemHelpers.text_message_outputs(res_1.new_items),
        ItemHelpers.text_message_outputs(res_2.new_items),
        ItemHelpers.text_message_outputs(res_3.new_items),
    ]

    translations = "\n\n".join(outputs)
    print(f"\n\nTranslations:\n\n{translations}")

    best_translation = await Runner.run(
        translation_picker,
        f"Input: {msg}\n\nTranslations:\n{translations}",
    )

    print("\n\n-----")

    print(f"Best translation: {best_translation.final_output}")

In [ ]:
asyncio.run(main())

Hi! Enter a message, and we'll translate it to Urdu.

Hello there


Translations:

آپ پر سلامتی ہو / آداب عرض ہے۔


آپ پر سلامتی ہو۔ (Aap par salamati ho.)


آپ کو سلام! / آداب!



-----
Best translation: The best translation is:

**آپ کو سلام! / آداب!**

Here's why:

*   **آپ کو سلام! (Aap ko salam!)** is a direct and common translation of "Hello." It's simple, widely understood, and appropriate in most situations.
*   **آداب! (Aadaab!)** is another very common and respectful greeting, similar to "Hello" or "Greetings."

The other options, "آپ پر سلامتی ہو / آداب عرض ہے۔" and "آپ پر سلامتی ہو۔ (Aap par salamati ho.)" are more akin to "Peace be upon you," which is a more formal and religious greeting than a simple "Hello."



## 4. Orchestrator Workers


In the orchestrator-workers workflow, a central LLM dynamically breaks down tasks, delegates them to worker LLMs, and synthesizes their results.

In [ ]:
from agents import MessageOutputItem

spanish_agent = Agent(
    name="spanish_agent",
    instructions="You translate the user's message to Spanish",
    handoff_description="An english to spanish translator",
    model=model
)

french_agent = Agent(
    name="french_agent",
    instructions="You translate the user's message to French",
    handoff_description="An english to french translator",
    model=model
)

italian_agent = Agent(
    name="italian_agent",
    instructions="You translate the user's message to Italian",
    handoff_description="An english to italian translator",
    model=model
)

orchestrator_agent = Agent(
    name="orchestrator_agent",
    instructions=(
        "You are a translation agent. You use the tools given to you to translate."
        "If asked for multiple translations, you call the relevant tools in order."
        "You never translate on your own, you always use the provided tools."
    ),
    tools=[
        spanish_agent.as_tool(
            tool_name="translate_to_spanish",
            tool_description="Translate the user's message to Spanish",
        ),
        french_agent.as_tool(
            tool_name="translate_to_french",
            tool_description="Translate the user's message to French",
        ),
        italian_agent.as_tool(
            tool_name="translate_to_italian",
            tool_description="Translate the user's message to Italian",
        ),
    ],
    model=model
)

synthesizer_agent = Agent(
    name="synthesizer_agent",
    instructions="You inspect translations, correct them if needed, and produce a final concatenated response. Always return the final response in with properly formatted translations.",
    model=model
)


async def main():
    msg = input("Hi! What would you like translated, and to which languages? ")

    # Run the entire orchestration in a single trace
    orchestrator_result = await Runner.run(orchestrator_agent, msg)

    for item in orchestrator_result.new_items:
        if isinstance(item, MessageOutputItem):
            text = ItemHelpers.text_message_output(item)
            if text:
                print(f"  - Translation step: {text}")

    synthesizer_result = await Runner.run(
        synthesizer_agent,
         (orchestrator_result.to_input_list() + [{"role": "user", 'content': f'pick best translation done for these {msg}' }])
    )

    print(f"\n\nFinal response:\n{synthesizer_result.final_output}")

    return synthesizer_result, orchestrator_result

In [ ]:
synth_result, orch_result = asyncio.run(main())

Hi! What would you like translated, and to which languages? Hello in Italian
  - Translation step: Ciao!



Final response:
The previous response "Ciao!" is a good translation of "Hello" in Italian.



In [ ]:
orch_result.to_input_list()

[{'content': 'Hello in Italian', 'role': 'user'},
 {'arguments': '{"input":"Hello"}',
  'call_id': '',
  'name': 'translate_to_italian',
  'type': 'function_call',
  'id': '__fake_id__'},
 {'call_id': '', 'output': 'Ciao!\n', 'type': 'function_call_output'},
 {'id': '__fake_id__',
  'content': [{'annotations': [], 'text': 'Ciao!\n', 'type': 'output_text'}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

In [ ]:
synth_result.final_output

'The previous response "Ciao!" is a good translation of "Hello" in Italian.\n'

## 5. Evaluator Optimizer

In the evaluator-optimizer workflow, one LLM call generates a response while another provides evaluation and feedback in a loop.

When to use this workflow: This workflow is particularly effective when we have clear evaluation criteria, and when iterative refinement provides measurable value. The two signs of good fit are, first, that LLM responses can be demonstrably improved when a human articulates their feedback; and second, that the LLM can provide such feedback. This is analogous to the iterative writing process a human writer might go through when producing a polished document.

In [ ]:
from __future__ import annotations
from typing import Literal
from dataclasses import dataclass
from agents import TResponseInputItem, trace

In [ ]:
story_outline_generator = Agent(
    name="story_outline_generator",
    instructions=(
        "You generate a very short story outline based on the user's input."
        "If there is any feedback provided, use it to improve the outline."
    ),
    model=model
)


@dataclass
class EvaluationFeedback:
    feedback: str
    score: Literal["pass", "needs_improvement", "fail"]


evaluator = Agent(
    name="evaluator",
    instructions=(
        "You evaluate a story outline and decide if it's good enough."
        "If it's not good enough, you provide feedback on what needs to be improved."
        "Never give it a pass on the first try."
    ),
    output_type=EvaluationFeedback,
    model=model
)

In [ ]:
async def main() -> None:
    msg = input("What kind of story would you like to hear? ")
    input_items: list[TResponseInputItem] = [{"content": msg, "role": "user"}]

    latest_outline: str | None = None

    while True:
        story_outline_result = await Runner.run(
            story_outline_generator,
            input_items,
        )

        input_items = story_outline_result.to_input_list()
        latest_outline = ItemHelpers.text_message_outputs(story_outline_result.new_items)
        print("Story outline generated")

        evaluator_result = await Runner.run(evaluator, input_items)
        result: EvaluationFeedback = evaluator_result.final_output

        print(f"Evaluator score: {result.score}")

        if result.score == "pass":
            print("Story outline is good enough, exiting.")
            break

        print("Re-running with feedback")

        input_items.append({"content": f"Feedback: {result.feedback}", "role": "user"})

    print(f"Final story outline: {latest_outline}")

In [ ]:
asyncio.run(main())

What kind of story would you like to hear? scifi
Story outline generated
Evaluator score: needs_improvement
Re-running with feedback
Story outline generated
Evaluator score: needs_improvement
Re-running with feedback
Story outline generated
Evaluator score: needs_improvement
Re-running with feedback
Story outline generated
Evaluator score: pass
Story outline is good enough, exiting.
Final story outline: Okay, I've further revised the outline based on your feedback, focusing on clarifying the mechanics and consequences to heighten the stakes:

*   **Setting:** A rust-colored desert Earth beneath twin moons, perpetually bathed in an eerie twilight. Colossal, petrified fungal blooms dominate the landscape, radiating residual "psionic resonance"—a lingering psychic echo from the bio-engineered crops that caused the ecological collapse. This resonance causes localized hallucinations and amplifies emotions.
*   **Character:** Elara, a former "Gardener" (bio-engineer) consumed by guilt over h